# Homework 1

For Cohort '26 of ML zoomcamp.

## Environment set up

For the homework, I'm managing the environment and packages with `uv`. Whilst I previously relied on `conda` data science workflows, I've transitioned to `uv` for its simplicity and speed, especially when only handling Python packages.

See below for the list of packages:

In [15]:
!uv pip list --quiet

Package                            Version
---------------------------------- ------------
aiohappyeyeballs                   2.7.1
aiohttp                            3.14.3
aiosignal                          1.4.0
aiosqlite                          0.22.1
alembic                            1.19.2
amplitude-analytics                1.2.3
annotated-doc                      0.0.5
annotated-types                    0.8.0
anyio                              4.15.1
appdirs                            1.4.4
appnope                            1.0.0
apprise                            1.13.1
asgi-lifespan                      2.1.0
asttokens                          3.0.2
asyncpg                            0.31.0
attrs                              26.1.0
beartype                           0.22.9
blinker                            1.9.0
burner-redis                       0.1.7
cachetools                         7.1.8
certifi                            2026.7.22
cffi                               2

## Question 1. Pandas version

The `pandas` version in the current environment is `3.0.5`.

In [16]:
import pandas as pd
print(pd.__version__)

3.0.5


## Getting the data

In [100]:
URL = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(URL)

## Question 2. Records count

There are 10,000 rows in the dataset.

In [101]:
print(len(df))

10000


## Question 3: Fuel types

There are three fuel types: Gasoline, Diesel, and Hybrid.

In [209]:
print(len(df.fuel_type.unique()))
print(df.fuel_type.unique())

3
<ArrowStringArray>
['Gasoline', 'Diesel', 'Hybrid']
Length: 3, dtype: str


## Question 4: Missing values

There are 2 columns, namely `horsepower`, and `acceleration` with missing values.

In [207]:
print((df.isna().sum() > 0).sum())
print(df.columns[(df.isna().sum() > 0)])

2
Index(['horsepower', 'acceleration'], dtype='str')


## Question 5. Max fuel efficiency

Process:

1. Quick look at the columns first to find the column that might store continents (`origin`)
2. Filter on "Asia" values
3. Run .max() on `fuel_efficiency_mpg`

The maximum fuel efficiency of Asian cars is 41.2.

In [104]:
df.columns

Index(['model_year', 'origin', 'fuel_type', 'drivetrain', 'num_doors',
       'engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight',
       'acceleration', 'fuel_efficiency_mpg'],
      dtype='str')

In [105]:
print(df[df.origin == "Asia"].fuel_efficiency_mpg.max())

41.2


## Question 6. Median value of horsepower

The median shifts lower upwards. Since we've replaced 877 observations with the modal value (which is lower than the median), the median is pulled down, naturally.

In [106]:
print(df.horsepower.median())

254.0


In [107]:
print(df.horsepower.mode()[0])

252.0


In [116]:
df.horsepower.isna().sum()

np.int64(877)

In [112]:
mode_hp = df.horsepower.mode()[0]
df_imp = df.copy()
df_imp.horsepower = df_imp.horsepower.fillna(mode_hp)

In [114]:
print(df_imp.horsepower.median())

252.0


## Question 7. Sum of weights

The weights are computed via the following (normal equation) formula:

$$w = (X^T X)^{-1}(X^Ty)$$

The weights vector `w` represent the coefficient estimates of linear regression. In other words, the slopes or parameters of the model. We can conclude that as the weight of the vehicle and the model year increase by one relative unit, the fuel efficiency increases by `0.136` and `0.233` units (engine displacement, presumably), respectively.

In [172]:
import numpy as np

df_asia = df_imp[df_imp.origin == "Asia"]
df_asia_two_cols = df_asia[["vehicle_weight", "model_year"]]
X = df_asia_two_cols.head(7).to_numpy()

**Computing the first half (Inverse Gram Matrix):**

This matrix captures the variance and covariance between the selected features. This is also why when two or more variables are multicollinear, it causes the matrix to collapse to (or near) zero, the inverse of which then causes the variance of the coefficients to explode, making the model parameters wildly unstable.

In [173]:
XTX = X.T.dot(X)

In [174]:
XTX_inv = np.linalg.inv(XTX)

**Computing the second half (Cross-Product Vector):** 

This measures the mathematical relationship between target `y` and the feature matrix `X`.

In [197]:
y = np.array([1100, 1300, 800, 900, 1000, 1100, 1200])
w = (XTX_inv @ X.T) @ y

In [200]:
print("Coefficient estimates:")
print(f"vehicle_weight: {w[0]}")
print(f"model_year: {w[1]}\n")
print(f"Sum of weights: {sum(w)}")

Coefficient estimates:
vehicle_weight: 0.13644776561569216
model_year: 0.23274920343358296

Sum of weights: 0.3691969690492751
